# 05 — Anxiety/Depression Circuit Analysis

Focused analysis of mechanosensitive ion channel expression in brain regions
relevant to anxiety and depression circuits:

- **Amygdala** (BLA, BMA, CeA, ATZ)
- **BNST**
- **Hippocampus** (CA1–CA4, DG, subiculum, parahippocampal)
- **Striatum** (caudate head/body/tail, putamen)
- **Nucleus Accumbens**
- **DLPFC** (MFG, SFG lateral)
- **Cingulate Cortex** (frontal, parietal, retrosplenial, subcallosal)
- **Thalamus** (anterior, medial, lateral, posterior, intralaminar, reticular, LGN, MGN)
- **Habenula** (lateral, medial)
- **Insula** (short, long gyri)
- **Hypothalamus** (AHA, ARH, DMH, LHA, LHM, LHT, PHA, PVH, VMH)

## Part 1: Subregion clustering
Are there notable differences within each region (e.g. BLA vs CeA)?

## Part 2: Region × cell-type clustering
Composite profiles (e.g. HPC-PV, HPC-SST, HPC-Pyramidal) clustered to reveal
cell-type-specific patterns across regions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler

from src import gene_lists, data_loader, cell_type_markers
from src.region_definitions import (
    SUBREGION_TO_PARENT, ALL_SUBREGIONS, PARENT_REGIONS,
    REGION_COLORS, get_label, get_parent,
)

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

## Load expression for target subregions

Average mechanosensitive channel expression across all 6 donors,
retaining subregion resolution.

In [ ]:
channel_genes = list(gene_lists.ALL_MECHANOSENSITIVE.keys())
region_expr = data_loader.get_microarray_region_means(channel_genes)

# Filter to our target subregions
target_cols = [c for c in region_expr.columns if c in ALL_SUBREGIONS]
subregion_expr = region_expr[target_cols].copy()

# Drop genes with too many NaN across these regions
subregion_expr = subregion_expr.dropna(axis=0, thresh=int(len(target_cols) * 0.7))
# Fill remaining NaN with row mean
subregion_expr = subregion_expr.T.fillna(subregion_expr.mean(axis=1)).T

print(f"Expression matrix: {subregion_expr.shape[0]} genes x {subregion_expr.shape[1]} subregions")
print(f"Regions covered: {sorted(set(get_parent(c) for c in subregion_expr.columns))}")

---
# Part 1: Subregion Clustering

Hierarchical clustering of all subregions to reveal within-region heterogeneity.

In [ ]:
# Z-score per gene
scaler = StandardScaler()
sub_z = pd.DataFrame(
    scaler.fit_transform(subregion_expr.values.T).T,
    index=subregion_expr.index,
    columns=subregion_expr.columns,
)

# Build annotation colors by parent region
col_colors = pd.Series(
    [REGION_COLORS.get(get_parent(c), '#95A5A6') for c in sub_z.columns],
    index=[get_label(c) for c in sub_z.columns],
    name='Region',
)

# Row colors by gene family
from src.clustering import build_family_palette
fam_palette = build_family_palette(list(sub_z.index))
row_colors = pd.Series(
    [fam_palette.get(gene_lists.get_family(g), '#95A5A6') for g in sub_z.index],
    index=[gene_lists.get_display_name(g) for g in sub_z.index],
    name='Family',
)

# Rename axes for display
plot_df = sub_z.copy()
plot_df.columns = [get_label(c) for c in plot_df.columns]
plot_df.index = [gene_lists.get_display_name(g) for g in plot_df.index]

g = sns.clustermap(
    plot_df,
    method='ward', metric='euclidean',
    row_colors=row_colors,
    col_colors=col_colors,
    cmap='RdBu_r', center=0,
    figsize=(24, 14),
    linewidths=0.3,
    xticklabels=True, yticklabels=True,
    dendrogram_ratio=(0.12, 0.12),
    cbar_pos=(0.02, 0.82, 0.03, 0.12),
)
g.ax_heatmap.tick_params(axis='x', rotation=90, labelsize=8)
g.ax_heatmap.tick_params(axis='y', labelsize=8)
g.fig.suptitle(
    'Mechanosensitive Ion Channels — Anxiety/Depression Subregion Clustering',
    fontsize=15, y=1.02,
)

# Add region legend
region_legend = [
    Patch(facecolor=REGION_COLORS[r], label=r)
    for r in PARENT_REGIONS if r in set(get_parent(c) for c in sub_z.columns)
]
g.ax_col_dendrogram.legend(
    handles=region_legend, title='Brain Region', fontsize=8,
    title_fontsize=9, loc='upper left', ncol=2,
)

plt.savefig('../figures/subregion_clustermap.png', dpi=150, bbox_inches='tight')
plt.show()

### Subregion dendrogram with cluster assignment

In [ ]:
# Cluster subregions
dist = pdist(sub_z.values.T, metric='euclidean')
Z = linkage(dist, method='ward')
labels = [get_label(c) for c in sub_z.columns]
label_colors = {get_label(c): REGION_COLORS.get(get_parent(c), '#95A5A6') for c in sub_z.columns}

fig, ax = plt.subplots(figsize=(22, 8))
dn = dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=9, ax=ax)

# Color leaf labels by parent region
xlbls = ax.get_xticklabels()
for lbl in xlbls:
    lbl.set_color(label_colors.get(lbl.get_text(), 'black'))
    lbl.set_fontweight('bold')

ax.set_title('Subregion Dendrogram — Mechanosensitive Channel Expression', fontsize=14)
ax.set_ylabel('Ward distance (Euclidean)', fontsize=12)

# Legend
ax.legend(
    handles=region_legend, title='Brain Region', fontsize=8,
    title_fontsize=9, loc='upper right', ncol=2,
)

plt.tight_layout()
plt.savefig('../figures/subregion_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

### Within-region heterogeneity

For each parent region with multiple subregions, compute the mean pairwise
distance between subregions to quantify internal heterogeneity.

In [ ]:
from scipy.spatial.distance import cdist

heterogeneity = {}
for parent in PARENT_REGIONS:
    subs = [c for c in sub_z.columns if get_parent(c) == parent]
    if len(subs) < 2:
        continue
    sub_data = sub_z[subs].values.T  # subregions x genes
    dists = pdist(sub_data, metric='euclidean')
    heterogeneity[parent] = {
        'mean_dist': np.mean(dists),
        'max_dist': np.max(dists),
        'n_subregions': len(subs),
    }

het_df = pd.DataFrame(heterogeneity).T.sort_values('mean_dist', ascending=False)
het_df.index.name = 'Region'

fig, ax = plt.subplots(figsize=(10, 6))
colors = [REGION_COLORS.get(r, '#95A5A6') for r in het_df.index]
bars = ax.barh(het_df.index, het_df['mean_dist'], color=colors, alpha=0.85)
ax.errorbar(
    het_df['mean_dist'], range(len(het_df)),
    xerr=het_df['max_dist'] - het_df['mean_dist'],
    fmt='none', ecolor='black', capsize=3, alpha=0.5,
)
ax.set_xlabel('Mean Pairwise Euclidean Distance (Z-scored expression)', fontsize=11)
ax.set_title('Within-Region Heterogeneity of Mechanosensitive Channels', fontsize=13)
ax.invert_yaxis()

# Annotate with n subregions
for i, (region, row) in enumerate(het_df.iterrows()):
    ax.text(row['mean_dist'] + 0.1, i, f'n={int(row["n_subregions"])}',
            va='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../figures/within_region_heterogeneity.png', dpi=150, bbox_inches='tight')
plt.show()

### Per-region subregion heatmaps

Heatmaps for regions with ≥3 subregions, showing FUS-core channel expression.

In [ ]:
core_genes = list(gene_lists.FUS_CORE_GENES.keys())
core_present = [g for g in core_genes if g in sub_z.index]

multi_sub_regions = [p for p in PARENT_REGIONS
                     if len([c for c in sub_z.columns if get_parent(c) == p]) >= 3]

n_panels = len(multi_sub_regions)
fig, axes = plt.subplots(n_panels, 1, figsize=(12, 3.5 * n_panels))
if n_panels == 1:
    axes = [axes]

for ax, parent in zip(axes, multi_sub_regions):
    subs = [c for c in sub_z.columns if get_parent(c) == parent]
    panel = sub_z.loc[core_present, subs]
    panel.columns = [get_label(c) for c in panel.columns]
    panel.index = [gene_lists.get_display_name(g) for g in panel.index]

    sns.heatmap(
        panel, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
        linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Z-score', 'shrink': 0.6},
    )
    ax.set_title(f'{parent} — FUS Core Channels', fontsize=12,
                 color=REGION_COLORS.get(parent, 'black'))
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('../figures/per_region_subregion_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 2: Region × Cell-Type Clustering

For each parent region, compute the spatial correlation between each
mechanosensitive channel gene and each neuron subtype marker set.
This gives us a composite profile like *HPC-PV*, *HPC-SST*, *AMY-Pyramidal*, etc.

We then cluster these composite profiles.

In [ ]:
# Load all needed genes (channels + neuron markers) at subregion level
neuron_markers = cell_type_markers.ALL_NEURON_MARKERS
all_genes_needed = list(set(channel_genes + neuron_markers))

all_region_expr = data_loader.get_microarray_region_means(all_genes_needed)
print(f"Full expression matrix: {all_region_expr.shape}")

In [ ]:
# For each parent region, average expression across its subregions
# to get one profile per parent region
parent_profiles = {}
for parent in PARENT_REGIONS:
    subs = [c for c in all_region_expr.columns if c in ALL_SUBREGIONS and get_parent(c) == parent]
    if subs:
        parent_profiles[parent] = all_region_expr[subs].mean(axis=1)

parent_expr = pd.DataFrame(parent_profiles)
print(f"Parent region expression: {parent_expr.shape[0]} genes x {parent_expr.shape[1]} regions")

# Split into channel and marker expression
ch_parent = parent_expr.loc[parent_expr.index.isin(channel_genes)].dropna()
mk_parent = parent_expr.loc[parent_expr.index.isin(neuron_markers)].dropna()
print(f"Channel genes: {ch_parent.shape[0]}, Marker genes: {mk_parent.shape[0]}")

In [ ]:
# For each parent region, compute neuron subtype enrichment
# using expression levels of marker genes within that region
# This creates region x cell-type composite scores

from scipy.stats import spearmanr

subtypes = list(cell_type_markers.NEURON_SUBTYPE_MARKERS.keys())

# Strategy: for each region, compute the mean expression of each
# channel gene, weighted by the relative expression of each cell-type
# marker in that region.
#
# Simpler approach: build a matrix of (region-celltype) x channel_gene
# where each entry is: channel_expr_in_region * marker_expr_in_region
# (interaction score)

composite_rows = []
composite_labels = []

for parent in PARENT_REGIONS:
    if parent not in parent_expr.columns:
        continue
    for subtype, markers in cell_type_markers.NEURON_SUBTYPE_MARKERS.items():
        present_markers = [m for m in markers if m in mk_parent.index]
        if not present_markers:
            continue

        # Mean marker expression in this region (proxy for cell-type abundance)
        marker_signal = mk_parent.loc[present_markers, parent].mean()

        # Channel expression in this region, scaled by marker signal
        ch_vals = ch_parent[parent].values

        # Composite = channel_expression * normalized_marker_signal
        composite_rows.append(ch_vals * marker_signal)
        composite_labels.append(f"{parent}\n{subtype}")

composite_df = pd.DataFrame(
    composite_rows,
    index=composite_labels,
    columns=ch_parent.index,
)

# Rename columns to display names
composite_df.columns = [gene_lists.get_display_name(g) for g in composite_df.columns]

print(f"Composite matrix: {composite_df.shape[0]} region-celltypes x {composite_df.shape[1]} channels")

In [ ]:
# Z-score across region-celltype combinations (per channel gene)
composite_z = composite_df.apply(lambda x: (x - x.mean()) / x.std(), axis=0)

# Build row annotation colors
row_region_colors = []
row_celltype_labels = []
for label in composite_z.index:
    parts = label.split('\n')
    region = parts[0]
    celltype = parts[1] if len(parts) > 1 else ''
    row_region_colors.append(REGION_COLORS.get(region, '#95A5A6'))
    row_celltype_labels.append(celltype)

row_colors_df = pd.DataFrame({
    'Region': row_region_colors,
}, index=composite_z.index)

# Column colors by gene family
gene_syms = list(ch_parent.index)
col_fam_colors = pd.Series(
    [fam_palette.get(gene_lists.get_family(g), '#95A5A6') for g in gene_syms],
    index=composite_z.columns,
    name='Family',
)

g = sns.clustermap(
    composite_z,
    method='ward', metric='euclidean',
    row_colors=row_colors_df,
    col_colors=col_fam_colors,
    cmap='RdBu_r', center=0,
    figsize=(22, 28),
    linewidths=0.2,
    xticklabels=True, yticklabels=True,
    dendrogram_ratio=(0.08, 0.08),
    cbar_pos=(0.02, 0.88, 0.02, 0.08),
)
g.ax_heatmap.tick_params(axis='x', rotation=90, labelsize=7)
g.ax_heatmap.tick_params(axis='y', labelsize=7)
g.fig.suptitle(
    'Region × Neuron Subtype — Mechanosensitive Channel Profiles',
    fontsize=15, y=1.01,
)

# Legend
g.ax_col_dendrogram.legend(
    handles=region_legend, title='Brain Region', fontsize=7,
    title_fontsize=8, loc='upper left', ncol=3,
)

plt.savefig('../figures/region_celltype_clustermap.png', dpi=150, bbox_inches='tight')
plt.show()

### Focused view: key neuron subtypes

Subset to the most relevant subtypes for anxiety/depression circuits:
PV, SST, VIP, Pyramidal, MSN-D1, MSN-D2.

In [ ]:
focus_subtypes = ['PV', 'SST', 'VIP', 'Pyramidal (upper layer)',
                  'Pyramidal (deep layer)', 'Medium Spiny (D1)',
                  'Medium Spiny (D2)', 'Serotonergic', 'Dopaminergic']

focus_mask = composite_z.index.map(
    lambda x: any(st in x for st in focus_subtypes)
)
focus_df = composite_z.loc[focus_mask]

# Also focus on FUS-core genes
core_display = [gene_lists.get_display_name(g) for g in core_present]
focus_cols = [c for c in focus_df.columns if c in core_display]
focus_df = focus_df[focus_cols]

# Row colors
focus_row_colors = pd.Series(
    [REGION_COLORS.get(idx.split('\n')[0], '#95A5A6') for idx in focus_df.index],
    index=focus_df.index,
    name='Region',
)

g2 = sns.clustermap(
    focus_df,
    method='ward', metric='euclidean',
    row_colors=focus_row_colors,
    cmap='RdBu_r', center=0,
    figsize=(14, 20),
    linewidths=0.4,
    xticklabels=True, yticklabels=True,
    annot=True, fmt='.1f',
    annot_kws={'fontsize': 6},
    dendrogram_ratio=(0.1, 0.1),
    cbar_pos=(0.02, 0.88, 0.02, 0.08),
)
g2.ax_heatmap.tick_params(axis='x', rotation=45, labelsize=9)
g2.ax_heatmap.tick_params(axis='y', labelsize=8)
g2.fig.suptitle(
    'Key Neuron Subtypes × FUS Core Channels — Region Comparison',
    fontsize=14, y=1.01,
)

plt.savefig('../figures/region_celltype_focused.png', dpi=150, bbox_inches='tight')
plt.show()

### Region × cell-type dendrogram

In [ ]:
# Dendrogram of the focused set
dist_focus = pdist(focus_df.values, metric='euclidean')
Z_focus = linkage(dist_focus, method='ward')

fig, ax = plt.subplots(figsize=(20, 10))
dn = dendrogram(
    Z_focus,
    labels=[idx.replace('\n', ' — ') for idx in focus_df.index],
    leaf_rotation=90, leaf_font_size=8, ax=ax,
)

# Color labels by region
xlbls = ax.get_xticklabels()
for lbl in xlbls:
    region_name = lbl.get_text().split(' — ')[0]
    lbl.set_color(REGION_COLORS.get(region_name, 'black'))
    lbl.set_fontweight('bold')

ax.set_title('Region × Cell-Type Dendrogram (FUS Core Channels)', fontsize=14)
ax.set_ylabel('Ward distance', fontsize=12)
ax.legend(
    handles=region_legend, title='Brain Region', fontsize=8,
    title_fontsize=9, loc='upper right', ncol=2,
)

plt.tight_layout()
plt.savefig('../figures/region_celltype_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

### Per-region cell-type profiles

For each region, show how FUS-core channel expression varies across neuron subtypes.

In [ ]:
# Extract per-region profiles for key subtypes
key_subtypes_short = ['PV', 'SST', 'VIP', 'Pyramidal (upper layer)',
                       'Pyramidal (deep layer)', 'Medium Spiny (D1)',
                       'Medium Spiny (D2)']

# Pick a subset of regions for clean visualization
focus_regions = ['Amygdala', 'Hippocampus', 'Striatum', 'Nucleus Accumbens',
                 'DLPFC', 'Cingulate Cortex', 'Thalamus', 'Insula']

n_reg = len(focus_regions)
fig, axes = plt.subplots(n_reg, 1, figsize=(14, 4 * n_reg))

for ax, region in zip(axes, focus_regions):
    # Get rows for this region and key subtypes
    region_rows = {}
    for st in key_subtypes_short:
        label = f"{region}\n{st}"
        if label in composite_df.index:
            region_rows[st] = composite_df.loc[label, core_display]

    if not region_rows:
        ax.text(0.5, 0.5, f'{region}: no data', ha='center', transform=ax.transAxes)
        continue

    panel = pd.DataFrame(region_rows).T
    # Z-score within this region for comparison
    panel_z = panel.apply(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else x * 0, axis=0)

    sns.heatmap(
        panel_z, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
        linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Z-score', 'shrink': 0.5},
    )
    ax.set_title(f'{region}', fontsize=12, fontweight='bold',
                 color=REGION_COLORS.get(region, 'black'))
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('../figures/per_region_celltype_profiles.png', dpi=150, bbox_inches='tight')
plt.show()